# 04 — Interaction chain: ewm-sm → Laya

**Question this notebook answers:** how does lattice state become a typed
context-policy decision?

The ewm-sm sidecar is a feature extractor; Laya is a typed-decision head.
This notebook runs `ingest`/`noether`/`bss` on a synthetic conversation,
renders the trajectory features into Laya's prose convention, and asks
Laya to choose the next context mode. Companion reference:
`docs/INTERACTION_MAP.md` §4.


In [ ]:
import os, sys

# make the `lab` package importable whether jupyter starts here or in the repo root
for _p in (os.getcwd(), os.path.dirname(os.getcwd())):
    if os.path.exists(os.path.join(_p, "lab")):
        sys.path.insert(0, _p)
        PROJ = _p
        break
else:
    raise RuntimeError("run jupyter from the project root or the notebooks/ directory")

from lab import EwmScene, LayaDaemon, BonsaiClient
print("project root:", PROJ)



In [ ]:
import json, os

WORK = os.path.join(PROJ, "work", "nb04")
os.makedirs(WORK, exist_ok=True)
union = os.path.join(WORK, "frames.jsonl")

turns = [
    ["tid1", "tid2", "tid3", "tid4", "tid5"],
    ["tid2", "tid3", "tid6", "tid7"],
    ["tid6", "tid7", "tid8", "tid9"],
    ["tid8", "tid9", "tid10"],
]
with open(union, "w") as fh:
    for i, tids in enumerate(turns, 1):
        fh.write(json.dumps({"id": i, "tokens": tids}) + "\n")

ewm = EwmScene()
print("binary:", ewm.bin)
print("frames:", json.dumps(turns))


## Step 1 — extract trajectory features from ewm-scene


In [ ]:
uni = ewm.ingest(union)
noe = ewm.noether(union)
bss = ewm.bss(union)

features = {
    "turn": len(uni["frames"]),
    "union_pop": uni["frames"][-1]["pop"],
    "key": uni["frames"][-1]["key"][:16],
    "full_memory_tids": len(turns[-1]),
    "novelty_tids": len({t for f in turns for t in f}),
    "last_dp": noe["dp"][-1],
    "last_ind1": round(noe["ind1"][-1], 3),
    "last_ind2": round(noe["ind2"][-1], 3),
    "last_ind3": round(noe["ind3"][-1], 3),
    "last_jaccard": round(bss["jaccard"][-1], 3),
}
print(json.dumps(features, indent=2))


## Step 2 — render the features as a prose state

Laya's encoder prefers natural language over raw structs (notebook 17 in
the research repo measured the improvement).


In [ ]:
laya = LayaDaemon()

state = LayaDaemon.prose_state(
    "What does the conversation add right now?",
    memory_tokens=features["full_memory_tids"],
    bss={"memory_coverage": features["last_jaccard"]},
)
print(state)


## Step 3 — ask Laya for the next context policy

One `choice` (the route) + one `noul` (`keep_short`) batched in a single
forward pass — exactly the request bonsai-ewm's `auto` advisor makes.


In [ ]:
decision = laya.route(
    state,
    options={
        "full": "keep the full materialized memory as the context",
        "compact": "keep only new, previously unseen tids as the context",
        "surprise": "keep only structurally surprising content (D-part plus Noether-flagged frames)",
    },
    instructions="How should the next context be built for the base model?",
    extra_noul={"keep_short": "The context should be kept short"},
    verbose=True,
)
print()
print("decision:", decision["decision"])
print("confidence:", round(decision["confidence"], 4))
print("keep_short:", round(decision["aux"]["keep_short"], 4))
print("input_tokens:", decision["input_tokens"])


## Step 4 — the same question, three different feature states

Watch how the typed decision follows the features: a crowded memory
should push `keep_short` up and favour `compact`; a memory full of
surprising transitions should favour `surprise`.


In [ ]:
scenarios = [
    ("idle",       LayaDaemon.prose_state("Say hello.", memory_tokens=4, bss={"coverage": 0.9})),
    ("redundant",  LayaDaemon.prose_state("Repeat the last answer.", memory_tokens=60, bss={"coverage": 0.97})),
    ("surprising", LayaDaemon.prose_state("The last answer changed the whole structure.", memory_tokens=60, bss={"coverage": 0.2})),
]
for name, state in scenarios:
    d = laya.route(
        state,
        options={"full": "whole memory", "compact": "new tids only", "surprise": "surprising frames only"},
        instructions="How should the next context be built?",
        extra_noul={"keep_short": "The context should be kept short"},
        verbose=False,
    )
    print(f"{name:11s} -> {d['decision']:9s} conf={d['confidence']:.3f} keep_short={d['aux']['keep_short']:.3f}")


In [ ]:
laya.close()
print('daemon closed')
